In [ ]:
!pip install -q kagglehub scikit-learn

import tensorflow as tf
import os
import numpy as np

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy('mixed_float16')

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

import kagglehub

from tensorflow.keras.applications import Xception, EfficientNetB4, ResNet50
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy('mixed_float16')

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

import warnings
warnings.filterwarnings("ignore")


TensorFlow: 2.19.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
import os
import numpy as np
from sklearn.model_selection import train_test_split
import tensorflow as tf
import kagglehub

dataset_path = kagglehub.dataset_download("kshitizbhargava/deepfake-face-images")
DATASET_PATH = os.path.join(dataset_path, "Final Dataset")

fake_dir = os.path.join(DATASET_PATH, "Fake")
real_dir = os.path.join(DATASET_PATH, "Real")

fake_images = [os.path.join(fake_dir, img) for img in os.listdir(fake_dir)]
real_images = [os.path.join(real_dir, img) for img in os.listdir(real_dir)]

X = fake_images + real_images
y = [0]*len(fake_images) + [1]*len(real_images)

X = np.array(X)
y = np.array(y)

print("Total images:", len(X))
print("Fake:", np.sum(y==0))
print("Real:", np.sum(y==1))


Using Colab cache for faster access to the 'deepfake-face-images' dataset.
Total images: 12890
Fake: 7000
Real: 5890


In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 8
EPOCHS = 5

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
)

print("Train:", len(X_train))
print("Val:", len(X_val))
print("Test:", len(X_test))

def process_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    img = img / 255.0
    return img, label

def create_dataset(X, y, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    ds = ds.map(process_image, num_parallel_calls=2)
    if shuffle:
        ds = ds.shuffle(1000)
    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(1)
    return ds

train_ds = create_dataset(X_train, y_train, shuffle=True)
val_ds   = create_dataset(X_val, y_val, shuffle=False)
test_ds  = create_dataset(X_test, y_test, shuffle=False)



Train: 9023
Val: 1933
Test: 1934


In [ ]:
from tensorflow.keras.applications import Xception, EfficientNetB0, ResNet50
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

def create_model(base_model_class):

    base_model = base_model_class(
        weights='imagenet',
        include_top=False,
        input_shape=(224,224,3)
    )

    for layer in base_model.layers[-30:]:
        layer.trainable = True

    x = GlobalAveragePooling2D()(base_model.output)
    x = Dense(256, activation='relu')(x)
    output = Dense(1, activation='sigmoid')(x)

    model = Model(base_model.input, output)

    model.compile(
        optimizer=Adam(1e-5),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model


In [ ]:
model_xception = create_model(Xception)
model_efficient = create_model(EfficientNetB4)
model_resnet = create_model(ResNet50)



In [ ]:
print("Training Xception...")
model_xception.fit(train_ds, validation_data=val_ds, epochs=EPOCHS)

print("Training EfficientNet...")
model_efficient.fit(train_ds, validation_data=val_ds, epochs=EPOCHS)

print("Training ResNet...")
model_resnet.fit(train_ds, validation_data=val_ds, epochs=EPOCHS)

Training Xception...
Epoch 1/5
1128/1128 ━━━━━━━━━━━━━━━━━━━━ 388s 191ms/step - accuracy: 0.7401 - loss: 0.5342 - val_accuracy: 0.9048 - val_loss: 0.2392
Epoch 2/5
1128/1128 ━━━━━━━━━━━━━━━━━━━━ 62s 54ms/step - accuracy: 0.8913 - loss: 0.2638 - val_accuracy: 0.9503 - val_loss: 0.1426
Epoch 3/5
1128/1128 ━━━━━━━━━━━━━━━━━━━━ 82s 54ms/step - accuracy: 0.9366 - loss: 0.1664 - val_accuracy: 0.9628 - val_loss: 0.1014
Epoch 4/5
1128/1128 ━━━━━━━━━━━━━━━━━━━━ 62s 54ms/step - accuracy: 0.9552 - loss: 0.1237 - val_accuracy: 0.9659 - val_loss: 0.0897
Epoch 5/5
1128/1128 ━━━━━━━━━━━━━━━━━━━━ 61s 53ms/step - accuracy: 0.9757 - loss: 0.0694 - val_accuracy: 0.9638 - val_loss: 0.0886
Training EfficientNet...
Epoch 1/5
1128/1128 ━━━━━━━━━━━━━━━━━━━━ 732s 331ms/step - accuracy: 0.6606 - loss: 0.6136 - val_accuracy: 0.8138 - val_loss: 0.4125
Epoch 2/5
1128/1128 ━━━━━━━━━━━━━━━━━━━━ 87s 76ms/step - accuracy: 0.8041 - loss: 0.4220 - val_accuracy: 0.8251 - val_loss: 0.3909
Epoch 3/5
1128/1128 ━━━━━━━━━━━━━

In [ ]:
def get_predictions(model, dataset):
    preds = model.predict(dataset, verbose=0)
    return preds.reshape(-1)

val_pred_x = get_predictions(model_xception, val_ds)
val_pred_e = get_predictions(model_efficient, val_ds)
val_pred_r = get_predictions(model_resnet, val_ds)

y_val = np.concatenate([y.numpy() for x, y in val_ds])


In [ ]:
from sklearn.linear_model import LogisticRegression

X_meta_train = np.vstack([val_pred_x, val_pred_e, val_pred_r]).T

meta_model = LogisticRegression(max_iter=1000)
meta_model.fit(X_meta_train, y_val)

print("Meta-classifier trained.")


Meta-classifier trained.


In [ ]:
test_pred_x = get_predictions(model_xception, test_ds)
test_pred_e = get_predictions(model_efficient, test_ds)
test_pred_r = get_predictions(model_resnet, test_ds)

y_test = np.concatenate([y.numpy() for x, y in test_ds])

X_meta_test = np.vstack([test_pred_x, test_pred_e, test_pred_r]).T

final_probs = meta_model.predict_proba(X_meta_test)[:,1]
final_preds = (final_probs > 0.5).astype(int)


In [ ]:
print("\n===== STACKING RESULTS =====")
print("Accuracy :", accuracy_score(y_test, final_preds))
print("Precision:", precision_score(y_test, final_preds, zero_division=0))
print("Recall   :", recall_score(y_test, final_preds, zero_division=0))
print("F1-score :", f1_score(y_test, final_preds, zero_division=0))
print("AUC      :", roc_auc_score(y_test, final_probs))



===== STACKING RESULTS =====
Accuracy : 0.9715615305067218
Precision: 0.9641657334826428
Recall   : 0.9739819004524887
F1-score : 0.9690489589195272
AUC      : 0.9961376858435682


In [ ]:
print("Unique test labels:", np.unique(y_test))
print("Unique final preds:", np.unique(final_preds))

Unique test labels: [0 1]
Unique final preds: [0 1]


In [ ]:
import tensorflow as tf

def add_gaussian_noise(image, label, std=0.1):
    noise = tf.random.normal(shape=tf.shape(image), mean=0.0, stddev=std)
    image = image + noise
    image = tf.clip_by_value(image, 0.0, 1.0)
    return image, label

def add_gaussian_blur(image, label):
    image = tf.image.random_brightness(image, 0.2)
    return image, label

def resolution_degradation(image, label, scale=0.5):
    original_size = tf.shape(image)[1]

    new_size = tf.cast(tf.cast(original_size, tf.float32) * scale, tf.int32)

    image = tf.image.resize(image, (new_size, new_size))

    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))

    return image, label


In [ ]:
test_noise = test_ds.map(lambda x,y: add_gaussian_noise(x,y))
test_blur = test_ds.map(lambda x,y: add_gaussian_blur(x,y))
test_res_50 = test_ds.map(lambda x,y: resolution_degradation(x,y,0.5))
test_res_25 = test_ds.map(lambda x,y: resolution_degradation(x,y,0.25))
test_res_12 = test_ds.map(lambda x,y: resolution_degradation(x,y,0.125))



In [ ]:
def evaluate_stacking(test_dataset, name):

    test_pred_x = get_predictions(model_xception, test_dataset)
    test_pred_e = get_predictions(model_efficient, test_dataset)
    test_pred_r = get_predictions(model_resnet, test_dataset)

    y_true = np.concatenate([y.numpy() for x,y in test_dataset])

    X_meta_test = np.vstack([test_pred_x, test_pred_e, test_pred_r]).T

    final_probs = meta_model.predict_proba(X_meta_test)[:,1]
    final_preds = (final_probs > 0.5).astype(int)

    print(f"\n===== {name} RESULTS =====")
    print("Accuracy :", accuracy_score(y_true, final_preds))
    print("Precision:", precision_score(y_true, final_preds, zero_division=0))
    print("Recall   :", recall_score(y_true, final_preds, zero_division=0))
    print("F1-score :", f1_score(y_true, final_preds, zero_division=0))
    print("AUC      :", roc_auc_score(y_true, final_probs))


In [ ]:
evaluate_stacking(test_ds, "CLEAN")
evaluate_stacking(test_noise, "GAUSSIAN NOISE")
evaluate_stacking(test_blur, "BLUR")
evaluate_stacking(test_res_50, "RESOLUTION 50%")
evaluate_stacking(test_res_25, "RESOLUTION 25%")
evaluate_stacking(test_res_12, "RESOLUTION 12.5%")

